In [1]:
# import pyomo.environ as pyo
from pyomo.dae import ContinuousSet, DerivativeVar, Simulator
import numpy as np
from pathlib import Path
# from metrics_logger import log_benchmark
from pyomo.contrib.parmest.experiment import Experiment
## This code simulates xdot = -a1x + a2e^(k1u1) + a3*e^(k2u2) - a4e^-x
#, a1,a2,a3,a4,k1,k2 are the unknown parameters that are to be estimated. 



# ========================
class TwoParameterExperiment(Experiment):
    def __init__(self, data, nfe, ncp):
        """
        Arguments
        ---------
        data: object containing vital experimental information
        nfe: number of finite elements
        ncp: number of collocation points for the finite elements
        """
        self.data = data
        self.nfe = nfe
        self.ncp = ncp
        self.model = None

        #############################
        # End constructor definition

    def get_labeled_model(self):
        if self.model is None:
            self.create_model()
            self.finalize_model()
            self.label_experiment()
        return self.model

    # Create flexible model without data
    def create_model(self):
        """
        This is an example user model provided to DoE library.
        It is a dynamic problem solved by Pyomo.DAE.

        Return
        ------
        m: a Pyomo.DAE model
        """

        m = self.model = pyo.ConcreteModel()

        # Model parameters
        m.w = pyo.Param(mutable=False, initialize=1.0)

        # Define model variables
        ########################
        # time
        m.t = ContinuousSet(bounds=[0, 1])

        # State and input
        m.x = pyo.Var(m.t, within=pyo.Reals)
        m.u1 = pyo.Var(m.t, within=pyo.Reals)
        m.u2 = pyo.Param(mutable=False, initialize=0.2)

        # Unknown parameter
        m.a1 = pyo.Var(within=pyo.Reals)
        m.a2 = pyo.Var(within=pyo.Reals)
        m.a3 = pyo.Var(within=pyo.Reals)
        m.a4 = pyo.Var(within=pyo.Reals)
        m.k1 = pyo.Var(within=pyo.Reals)
        m.k2 = pyo.Var(within=pyo.Reals)

        # Differential variables wrt x
        m.dxdt = DerivativeVar(m.x, wrt=m.t)
       

        ########################
        # End variable def.

        # Equation definition
        ########################

        # Expression for the state evolution

        # State odes
        @m.Constraint(m.t)
        def x_ode(m, t):
            return m.dxdt[t] == -m.a1 * m.x[t] + m.a2* pyo.exp(m.k1*m.u1[t]) + m.a3*pyo.exp(m.k2*m.u2) - m.a4*pyo.exp(-m.x[t])
            


        ########################
        # End equation definition

    def finalize_model(self):
        """
        Example finalize model function. There are two main tasks
        here:

            1. Extracting useful information for the model to align
               with the experiment. (Here: CA0, t_final, t_control)
            2. Discretizing the model subject to this information.

        """
        m = self.model

        # Unpacking data before simulation
        control_points = self.data["control_points"]

        # Set initial state values for the experiment
        m.x[0].value = self.data["x0"]

        # Update model time `t` with time range and control time points
        m.t.update(self.data["t_range"])
        m.t.update(control_points)

        # Fix the unknown parameter values
        m.a1.fix(self.data["a1"])
        m.a2.fix(self.data["a2"])
        m.a3.fix(self.data["a3"])
        m.a4.fix(self.data["a4"])
        m.k1.fix(self.data["k1"])
        m.k2.fix(self.data["k2"])

        # Add upper and lower bounds to the design variable, CA[0]
        m.x[0].setlb(self.data["x_bounds"][0])
        m.x[0].setub(self.data["x_bounds"][1])

        m.t_control = control_points

        # Discretizing the model
        discr = pyo.TransformationFactory("dae.collocation")
        discr.apply_to(m, nfe=self.nfe, ncp=self.ncp, wrt=m.t)

        # Initializing control input in the model
        cv = None
        for t in m.t:
            if t in control_points:
                cv = control_points[t]
                m.u1[t].fix()
            m.u1[t].setlb(self.data["u_bounds"][0])
            m.u1[t].setub(self.data["u_bounds"][1])
            m.u1[t] = cv

        # Make a constraint that holds control input constant between control time points
        @m.Constraint(m.t - control_points)
        def u_control(m, t):
            """
            Piecewise constant control input between control points
            """
            neighbour_t = max(tc for tc in control_points if tc < t)
            return m.u1[t] == m.u1[neighbour_t]


        

        #########################
        # End model finalization

    def label_experiment(self):
        """
        Example for annotating (labeling) the model with a
        full experiment.
        """
        m = self.model

        # Set measurement labels
        m.experiment_outputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        # Add CA to experiment outputs
        m.experiment_outputs.update((m.x[t], None) for t in m.t_control)


        # Adding error for measurement values (assuming no covariance and constant error for all measurements)
        m.measurement_error = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        meas_error = 1e-2  # Error in state measurement
        # Add measurement error for CA
        m.measurement_error.update((m.x[t], meas_error) for t in m.t_control)
   

        # Identify design variables (experiment inputs) for the model
        m.experiment_inputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        # Add experimental input label for initial state
        m.experiment_inputs[m.x[m.t.first()]] = None
        # Add experimental input label for control input
        m.experiment_inputs.update((m.u1[t], None) for t in m.t_control)

        # Add unknown parameter labels
        m.unknown_parameters = pyo.Suffix(direction=pyo.Suffix.LOCAL)
        # Add labels to all unknown parameters with nominal value as the value
        m.unknown_parameters.update((k, pyo.value(k)) for k in [m.a1, m.a2, m.a3, m.a4, m.k1, m.k2])

        #########################
        # End model labeling

In [2]:
from pyomo.common.dependencies import numpy as np

from pyomo.contrib.doe import DesignOfExperiments

import pyomo.environ as pyo
from pyomo.environ import (
    ConcreteModel,
    Var,
    Param,
    Constraint,
    TransformationFactory,
    SolverFactory,
    Objective,
    minimize,
    value as pyovalue,
    Suffix,
    Expression,
    sin,
    NonNegativeReals,
)



data_ex = {"x0": 1, "x_bounds": [0, 10], "t_range": [0, 1],
           "control_points": {"0": -1, "0.125": 1, "0.25": 1, "0.375": 1, "0.5": 1, "0.625": 1, "0.75": 1,
                              "0.875": 1, "1": 1}, "u_bounds": [-1.0, 1.0], "a1": 1.0, "a2":0.2, "a3": 0.2, "a4": 1.0, "k1": 0.1, "k2": 1.0}
# Put control input control time points into correct format for two parameter experiment
data_ex["control_points"] = {
    float(k): v for k, v in data_ex["control_points"].items()
}

# Create a two parameterExperiment object; data and discretization information are part
# of the constructor of this object
experiment = TwoParameterExperiment(data=data_ex, nfe=10, ncp=3)

# Use a central difference, with step size 1e-3
fd_formula = "central"
step_size = 1e-3

# Use the determinant objective with scaled sensitivity matrix
objective_option = "determinant"
scale_nominal_param_value = True

# Create the DesignOfExperiments object
# We will not be passing any prior information in this example.
# We also will rely on the initialization routine within
# the DesignOfExperiments class.
doe_obj = DesignOfExperiments(
    experiment,
    fd_formula=fd_formula,
    step=step_size,
    objective_option=objective_option,
    scale_constant_value=1,
    scale_nominal_param_value=scale_nominal_param_value,
    prior_FIM=None,
    jac_initial=None,
    fim_initial=None,
    L_diagonal_lower_bound=1e-7,
    solver=SolverFactory('IPOPT', options={'linear_solver': 'mumps'}),
    tee= True,
    get_labeled_model_args=None,
    _Cholesky_option=True,
    _only_compute_fim_lower=True,
)

# # Make design ranges to compute the full factorial design
# design_ranges = {"x[0]": [0.5, 1.5, 20], "u[0]": [-1, 1, 20\]}

# # Compute the full factorial design with the sequential FIM calculation
# doe_obj.compute_FIM_full_factorial(design_ranges=design_ranges, method="sequential")

# # Plot the results
# doe_obj.draw_factorial_figure(
#     sensitivity_design_variables=["x[0]", "u[0]"],
#     fixed_design_variables={
#         "u[0.125]": 1,
#         "u[0.25]": 1,
#         "u[0.375]": 1,
#         "u[0.5]": 1,
#         "u[0.625]": 1,
#         "u[0.75]": 1,
#         "u[0.875]": 1,
#         "u[1]": 1,
#     },
#     title_text="Single State Example",
#     xlabel_text="State",
#     ylabel_text="Input",
#     figure_file_name="example_one_state_compute_FIM",
#     log_scale=False,
# )


doe_obj.run_doe()

# Print out a results summary
print("Optimal experiment values: ")
print(
    "\tInitial state: {:.2f}".format(
        doe_obj.results["Experiment Design"][0]
    )
)
print(
    ("\t Control input values: [" + "{:.2f}, " * 8 + "{:.2f}]").format(
        *doe_obj.results["Experiment Design"][1:]
    )
)
print("FIM at optimal design:\n {}".format(np.array(doe_obj.results["FIM"])))
print(
    "Objective value at optimal design: {:.2f}".format(
        pyo.value(doe_obj.model.objective)
    )
)

print(doe_obj.results["Experiment Design Names"])

#print(sorted(doe_obj.results.keys()))

print("Solve time (s):", doe_obj.results["Solve Time"])
print("Build time (s):", doe_obj.results["Build Time"])
print("Initialization time (s):", doe_obj.results["Initialization Time"])
print("Total wall time (s):", doe_obj.results["Wall-clock Time"])

###################
# End optimal DoE



Ipopt 3.13.2: linear_solver=mumps


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt

This version of Ipopt was compiled from source code available at
    https://github.com/IDAES/Ipopt as part of the Institute for the Design of
    Advanced Energy Systems Process Systems Engineering Framework (IDAES PSE
    Framework) Copyright (c) 2018-2019. See https://github.com/IDAES/idaes-pse.

This version of Ipopt was compiled using HSL, a collection of Fortran codes
    for large-scale scientific computation.  All technical papers, sales and
    publicity material resulting from use of the HSL codes within IPOPT must
    contain the following acknowledgement:
        HSL, a collection of Fortran codes for large-scale scientific
        comp

ValueError: Cannot load a SolverResults object with bad status: error

In [ ]:
# from metrics_logger import log_benchmark

# NOTEBOOK_ID = "6"
# scenario = "central"
# log_benchmark(
#     notebook_id = NOTEBOOK_ID,
#     scenario = scenario,
#     solve_time_s = doe_obj.results["Solve Time"],
#     build_time_s = doe_obj.results["Build Time"],
#     init_time_s = doe_obj.results["Initialization Time"],
#     total_wall_time_s = doe_obj.results["Wall-clock Time"]
# )

In [ ]:
# import pyomo.environ as pyo
# from pyomo.dae import ContinuousSet, DerivativeVar, Simulator
# from pyomo.contrib.parmest.experiment import Experiment
# from pyomo.contrib.doe import DesignOfExperiments
# import numpy as np

# # ----------------------------------
# # helpers
# # ----------------------------------
# def piecewise_const_from_cp(control_points):
#     cps = sorted((float(t), float(v)) for t, v in control_points.items())
#     ts = [t for t, _ in cps]
#     vs = [v for _, v in cps]
#     def u_of_t(t):
#         idx = max(i for i, tc in enumerate(ts) if tc <= t)
#         return vs[idx]
#     return u_of_t

# def interp_to_grid(tsim, xsim, grid):
#     return np.interp(np.array(list(grid), dtype=float), np.array(tsim), np.array(xsim))


# # ======================================================
# # Experiment (correct: control at CPs are free variables)
# # ======================================================
# class TwoParameterExperiment(Experiment):
#     def __init__(self, data, nfe, ncp):
#         self.data = data
#         self.nfe = nfe
#         self.ncp = ncp
#         self.model = None

#     def get_labeled_model(self):
#         if self.model is None:
#             self.create_model()
#             self._warm_start_with_simulation()  # generate good initial trajectory
#             self.finalize_model()                # discretize + constraints (no fixing u1 at CPs)
#             self.label_experiment()
#         return self.model

#     # ----------------------------
#     # base continuous model
#     # ----------------------------
#     def create_model(self):
#         m = self.model = pyo.ConcreteModel()

#         t0, tf = self.data["t_range"]
#         m.t = ContinuousSet(bounds=[t0, tf])

#         m.x  = pyo.Var(m.t, within=pyo.Reals)
#         m.u1 = pyo.Var(m.t, within=pyo.Reals)
#         m.u2 = pyo.Param(mutable=False, initialize=0.2)

#         m.a1 = pyo.Var(initialize=self.data["a1"])
#         m.a2 = pyo.Var(initialize=self.data["a2"])
#         m.a3 = pyo.Var(initialize=self.data["a3"])
#         m.a4 = pyo.Var(initialize=self.data["a4"])
#         m.k1 = pyo.Var(initialize=self.data["k1"])
#         m.k2 = pyo.Var(initialize=self.data["k2"])

#         m.dxdt = DerivativeVar(m.x, wrt=m.t)

#         @m.Constraint(m.t)
#         def x_ode(m, t):
#             return m.dxdt[t] == (
#                 -m.a1 * m.x[t]
#                 + m.a2 * pyo.exp(m.k1 * m.u1[t])
#                 + m.a3 * pyo.exp(m.k2 * m.u2)
#                 - m.a4 * pyo.exp(-m.x[t])
#             )

#         m.x[t0].value = self.data["x0"]

#     # ----------------------------
#     # warm-start via simulation
#     # ----------------------------
#     def _warm_start_with_simulation(self):
#         m = self.model
#         cp = self.data["control_points"]

#         # fix params for simulation
#         for v in (m.a1, m.a2, m.a3, m.a4, m.k1, m.k2):
#             v.fix(pyo.value(v))

#         sim = Simulator(m, package="scipy")
#         u1_fun = piecewise_const_from_cp(cp)

#         tsim, prof = sim.simulate(
#             numpoints=max(200, self.nfe * max(self.ncp, 1) * 10),
#             integrator="vode",
#             varying_inputs={m.u1: u1_fun},
#             initial_condition={m.x: self.data["x0"]},
#         )

#         self._warm_t = np.array(tsim, dtype=float)
#         self._warm_x = np.array(prof[m.x], dtype=float)

#         for v in (m.a1, m.a2, m.a3, m.a4, m.k1, m.k2):
#             v.unfix()

#     # ----------------------------
#     # discretize & impose PW-constant control (but keep CP vars free!)
#     # ----------------------------
#     def finalize_model(self):
#         m = self.model
#         data = self.data
#         cp = data["control_points"]
#         t0, tf = data["t_range"]

#         m.x.setlb(data["x_bounds"][0])
#         m.x.setub(data["x_bounds"][1])

#         # parameters fixed at nominal for DoE
#         m.a1.fix(data["a1"]); m.a2.fix(data["a2"]); m.a3.fix(data["a3"]); m.a4.fix(data["a4"])
#         m.k1.fix(data["k1"]); m.k2.fix(data["k2"])

#         # add control point times into time set & store as Set
#         m.t.update(cp.keys())
#         m.t_control = pyo.Set(initialize=sorted(float(t) for t in cp.keys()), ordered=True)

#         # discretize with FD (robust)
#         discr = pyo.TransformationFactory("dae.finite_difference")
#         discr.apply_to(m, nfe=self.nfe, scheme="BACKWARD", wrt=m.t)

#         # bounds for control everywhere
#         for t in m.t:
#             m.u1[t].setlb(data["u_bounds"][0])
#             m.u1[t].setub(data["u_bounds"][1])

#         # initialize u1 values (do NOT fix at control points)
#         # carry last control value between CPs; CPs remain free design vars
#         last_cv = None
#         for t in sorted(float(tt) for tt in m.t):
#             if t in cp:
#                 last_cv = cp[t]
#                 # m.u1[t] remains a free variable; set a good start only
#                 m.u1[t].value = last_cv
#             else:
#                 if last_cv is not None:
#                     m.u1[t].value = last_cv

#         # piecewise-constant: non-CP times equal to left CP variable
#         @m.Constraint(m.t - m.t_control)
#         def u_control(m, t):
#             left = max(tc for tc in m.t_control if tc < float(t))
#             return m.u1[t] == m.u1[left]

#         # inject warm-start trajectory for x
#         grid = list(m.t)
#         xgrid = interp_to_grid(self._warm_t, self._warm_x, grid)
#         for t_val, x_val in zip(grid, xgrid):
#             m.x[t_val].value = float(np.clip(x_val, data["x_bounds"][0], data["x_bounds"][1]))

#     # ----------------------------
#     # labels for DoE
#     # ----------------------------
#     def label_experiment(self):
#         m = self.model
#         # outputs at control times
#         m.experiment_outputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
#         m.experiment_outputs.update((m.x[t], None) for t in m.t_control)

#         # measurement error
#         m.measurement_error = pyo.Suffix(direction=pyo.Suffix.LOCAL)
#         m.measurement_error.update((m.x[t], 1e-2) for t in m.t_control)

#         # design variables: initial state + control at control points
#         m.experiment_inputs = pyo.Suffix(direction=pyo.Suffix.LOCAL)
#         m.experiment_inputs[m.x[m.t.first()]] = None              # x(0)
#         m.experiment_inputs.update((m.u1[t], None) for t in m.t_control)  # u1 at CPs (FREE)

#         # unknown parameters (nominals)
#         m.unknown_parameters = pyo.Suffix(direction=pyo.Suffix.LOCAL)
#         m.unknown_parameters.update(
#             (k, pyo.value(k))
#             for k in [m.a1, m.a2, m.a3, m.a4, m.k1, m.k2]
#         )


# # ======================================================
# # problem data (stabilized)
# # ======================================================
# data_ex = {
#     "x0": 1.0,
#     "x_bounds": [0.01, 5.0],
#     "t_range": [0.0, 2.0],
#     "control_points": {
#         0.0: -1, 0.125: 1, 0.25: -1, 0.375: 1,
#         0.5: -1, 0.625: 1, 0.75: -1, 0.875: 1, 1.0: -1,
#     },
#     "u_bounds": [-1.0, 1.0],
#     "a1": 0.8, "a2": 0.4, "a3": 0.1, "a4": 0.5,
#     "k1": 0.3, "k2": 0.5,
# }

# # ======================================================
# # build & solve DoE
# # ======================================================
# experiment = TwoParameterExperiment(data=data_ex, nfe=12, ncp=3)

# ipopt = pyo.SolverFactory("ipopt")
# ipopt.options.update({
#     "tol": 1e-8,
#     "acceptable_tol": 1e-6,
#     "max_iter": 3000,
#     "mu_strategy": "adaptive",
#     "hessian_approximation": "limited-memory",
#     "linear_solver": "mumps",  # switch to "ma57" if available
#     "bound_push": 1e-6,
#     "bound_frac": 1e-6,
# })

# doe_obj = DesignOfExperiments(
#     experiment,
#     fd_formula="forward",          # start robust; switch to "central" later if you want
#     step=1e-4,
#     objective_option="determinant",
#     scale_constant_value=1,
#     scale_nominal_param_value=True,
#     prior_FIM=None,
#     L_diagonal_lower_bound=1e-6,
#     solver=ipopt,
#     tee=True,
#     _Cholesky_option=True,
#     _only_compute_fim_lower=True,
# )

# # run
# doe_obj.run_doe()

# # report
# print("\n===== Optimal Design Results =====")
# print("Initial state: {:.3f}".format(doe_obj.results["Experiment Design"][0]))
# print("Control input values:")
# print(np.round(doe_obj.results["Experiment Design"][1:], 3))
# print("\nFIM at optimal design:\n", np.array(doe_obj.results["FIM"]))
# print("\nObjective value:", pyo.value(doe_obj.model.objective))
# print("Solve time (s):", doe_obj.results["Solve Time"])
# print("Build time (s):", doe_obj.results["Build Time"])
# print("Initialization time (s):", doe_obj.results["Initialization Time"])
# print("Total wall time (s):", doe_obj.results["Wall-clock Time"])
